In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import xgboost as xgb
from xgboost import plot_importance
from sklearn import metrics
from sklearn.metrics import r2_score, mean_squared_error,mean_absolute_error
import matplotlib.pyplot as plt
import shap
from umap import UMAP
from joypy import joyplot
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import cmaps
from sklearn.cluster import DBSCAN
from sklearn.model_selection import KFold
from tqdm import tqdm
from itertools import combinations
from joblib import Parallel, delayed
import os
import matplotlib.colors as mcolors
from sklearn.cluster import DBSCAN
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
import palettable
import shutil
from shap_utiles import  grouped_heatmap
import cartopy.feature as cfeature
from scipy.stats import gaussian_kde, linregress
from sklearn.neighbors import BallTree
from sklearn.cluster import AgglomerativeClustering
from scipy import stats
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils.dataframe import dataframe_to_rows
from scipy.stats import pearsonr
import os
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator, LinearLocator
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from scipy.ndimage.filters import gaussian_filter
import matplotlib.path as mpath
import cartopy.crs as ccrs
plt.rc('font', family='Times New Roman')

d:\anaconda\envs\shap\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\84648\AppData\Local\Temp\ipykernel_20780\1490927240.py:47: DeprecationWarning: Please import `gaussian_filter` from the `scipy.ndimage` namespace; the `scipy.ndimage.filters` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.filters import gaussian_filter


In [2]:
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.colors as mcolors
def air_mass_cmap():
    # 定义颜色节点：深褐色 -> 浅褐色 -> 浅蓝色 -> 白色
    colors = ['#FFFFFF',"#ADDFAD","#174892","#0B0D7A","#280550"]
    
    # 创建颜色映射
    cmap = mcolors.LinearSegmentedColormap.from_list('CAO', colors)
    return cmap
def dry_wet_colormap():
    # 定义颜色节点：深褐色 -> 浅褐色 -> 浅蓝色 -> 白色
    colors = ["#572B0B","#AD5618", '#D2B48C', "#BCE7E7", '#FFFFFF']
    
    # 创建颜色映射
    cmap = mcolors.LinearSegmentedColormap.from_list('dry_wet', colors)
    return cmap
def load_feature_data(feature_names,chosed_class,stage):
    # 基础路径
    ano_path = r'X:\feature_distri\class_ano'
    p_path = r'X:\feature_distri\class_p'
    distri_path = r'X:\feature_distri\distri'
    propation_path = r'X:\feature_distri\propagation'
    # 初始化数据集
    ano_ds = xr.Dataset()
    p_ds = xr.Dataset()
    # 加载分布数据
    distri = xr.open_dataarray(os.path.join(distri_path, f"{chosed_class}_{stage}_distri.nc"))
    propation = pd.read_excel(os.path.join(propation_path,f'{chosed_class}_whole_propagation.xlsx'))
    # 处理每个特征
    for feature in feature_names:
        # 加载距平数据
        ano_data = xr.open_dataarray(os.path.join(ano_path, f"{chosed_class}_{feature}_{stage}.nc"))
        ano_ds[feature] = ano_data
        # 加载p值数据（z500mean除外）
        p_data = xr.open_dataarray(os.path.join(p_path, f"{chosed_class}_{feature}_{stage}.nc"))
        p_ds[feature] = p_data
        
    # 风场的检验,取最小P值
    if 'u500' in p_ds and 'v500' in p_ds:
        wind_500_p = xr.where(p_ds['u500'] < p_ds['v500'], p_ds['u500'], p_ds['v500'])
        p_ds['wind_500'] = wind_500_p
        p_ds = p_ds.drop_vars(['u500', 'v500'])
    # 925hPa风场
    if 'u925' in p_ds and 'v925' in p_ds:
        wind_925_p = xr.where(p_ds['u925'] < p_ds['v925'], p_ds['u925'], p_ds['v925'])
        p_ds['wind_925'] = wind_925_p
        p_ds = p_ds.drop_vars(['u925', 'v925'])
    if 'u_avg' in p_ds and 'v_avg' in p_ds:
        wind_925_p = xr.where(p_ds['u_avg'] < p_ds['v_avg'], p_ds['u_avg'], p_ds['v_avg'])
        p_ds['wind_avg'] = wind_925_p
        p_ds = p_ds.drop_vars(['u_avg', 'v_avg'])
    distri.values = gaussian_filter(distri,3)
    return distri,propation,ano_ds, p_ds
def load_feature_mean(feature_names,chosed_class,stage):
    # 基础路径
    path = r'X:\feature_distri\class_composite'
    data = xr.open_dataset(os.path.join(path, f"{chosed_class}_{stage}.nc"))
    return data[feature_names]


In [3]:
def basic_draw_sets(ax):
    extent = [-45, 80,50, 86]
    left,right,down,upper = extent
    #ax.add_feature(cfeature.COASTLINE.with_scale('110m'))
    ax.set_extent(extent, ccrs.PlateCarree())
    #经纬度网格线
    gl = ax.gridlines(
        xlocs=np.arange(left, right, 30), ylocs=np.arange(down, upper,10),
        draw_labels=True, x_inline=False, y_inline=False,rotate_labels=False,
        linewidth=1, linestyle='--', color='gray',zorder=99999)
    gl.top_labels = False
    gl.xlabel_style = {'size': 18}   # 经度标签字体大小
    gl.ylabel_style = {'size': 18}   # 纬度标签字体大小
    rect = mpath.Path([[left, down], [right, down],  [right, upper],  [left, upper],  [left, down],  ]).interpolated(20)  # 将路径点插值密集一些，会更平滑
    # 转换路径的坐标系
    proj_to_data = ccrs.PlateCarree()._as_mpl_transform(ax) - ax.transData  # 建立坐标投影的转换（使用时仅需修改ax变量为需要作用的子图对象）
    rect_in_target = proj_to_data.transform_path(rect)  # 将创建好的路径进行坐标转换（使用时仅需修改rect参数为生成的path对象）
    # 设置GeoAxes边界
    ax.set_boundary(rect_in_target) 
def creat_cax(fig,ax):
    #手动指定坐标区域
    chartBox = ax.get_position() 
    x, y, w, h = chartBox.x0, chartBox.y0, chartBox.width, chartBox.height 
    cax = fig.add_axes([x, y-0.03, w, 0.01])
    return cax

def draw_CAO(ax,distri,var,p_values):
    colors = [ "#EEFF00", "#F83200"]
    # 创建自定义colormap
    custom_gray_cmap = mcolors.LinearSegmentedColormap.from_list('custom_gray', colors[::-1], N=256)

    p_level  = 0.01
    #冷空气爆发和感热输送
    theta_trop,u925,v925,CAO_depth= gaussian_filter(var.theta_trop.values,3),var.u925,var.v925,gaussian_filter(var.CAO_depth,1)
    CAO_depth_p,wind925_p,theta_trop_p = p_values.CAO_depth,p_values.wind_925,p_values.theta_trop
    lons,lats = var.longitude,var.latitude
    #基础绘图设置
    ax.add_feature(cfeature.LAND, edgecolor='black',facecolor='lightgray',linewidth=1,zorder=1)
    basic_draw_sets(ax)
    #对流层顶位温-----------------------------------------------
    theta_trop_levels = np.arange(-12,0,4)
    theta_trop[theta_trop_p>p_level] = np.nan
    CAO_depth[CAO_depth_p>p_level] = np.nan
    ax.contour(lons,lats,theta_trop,levels=theta_trop_levels,extend='both',cmap=custom_gray_cmap,alpha=1, transform =ccrs.PlateCarree(),zorder=2,linewidths=2)
    #cb = ax.clabel(CS,fontsize=18, fmt='%d',inline=1,colors='k') #给等值线标注上数字)
    #冷空气爆发
    C1 = ax.contourf(lons,lats,CAO_depth,levels=np.arange(0,171,10),extend='both',cmap= air_mass_cmap() ,zorder=0, transform =ccrs.PlateCarree())#画等值线
    distri = distri.sel(latitude=u925.latitude,longitude=u925.longitude)
    sig_mask_925 =  (wind925_p < p_level)&(distri.values>10)
    u925, v925 = np.where(sig_mask_925, u925, np.nan), np.where(sig_mask_925, v925, np.nan)
    # 绘制箭头（quiver）
    xd,yd = 14,6
    Q = ax.quiver(lons[::xd],lats[::yd],u925[::yd,::xd], v925[::yd,::xd],color="#FF0000",pivot="middle",scale=60, width=0.005,transform=ccrs.PlateCarree(),alpha=1,zorder=11)
    # 添加 quiverkey（axes 右上角）
    ax.quiverkey(Q, 0.85, 0.92, 5, "5 m/s",coordinates="axes", color="#FF0000",fontproperties={"size": 14})
    #感热
    ax.coastlines(color='black',linewidth=1)
    return C1
def draw_trickness_gra(ax,var,p_values):
    p_level  = 0.01
    #冷空气爆发和感热输送
    #------------------------------------------------------
    pbh = gaussian_filter(var.pbh.values,1)
    pbh_p = p_values.pbh
    pbh[pbh_p>p_level] = np.nan
    lons,lats = var.longitude,var.latitude
    #地理数据
    #基础绘图设置
    ax.add_feature(cfeature.LAND, edgecolor='black',facecolor='lightgray',linewidth=1,zorder=2)
    basic_draw_sets(ax)
    C1 = ax.contourf(lons,lats,pbh,levels=np.arange(0,200+1,20),extend='both',cmap= cmaps.MPL_BuPu,zorder=0,alpha=1, transform =ccrs.PlateCarree())#画等值线
    ax.coastlines(color='black',linewidth=1)
    return C1
def draw_q(ax,mean_vars,var,distri,p_values):
    p_level  = 0.01
    q,siconc= var.q.values,gaussian_filter(var.siconc.values,1)
    q_p,siconc_p = p_values.q.values,p_values.siconc.values
    lons,lats = var.longitude,var.latitude
    siconc[siconc_p>p_level] = np.nan
    q[q_p>p_level] = np.nan
    basic_draw_sets(ax)
    C1 = ax.contourf(lons,lats,q,levels=np.arange(-0.6,0.05,0.05),extend='both',cmap= dry_wet_colormap(),zorder=0, transform =ccrs.PlateCarree())#画等值线
    ax.coastlines(color='black',linewidth=1)
    z500,u500,v500 = mean_vars.z500,mean_vars.u500,mean_vars.v500
    z500_levels = np.arange(5050,5450,50)
    CS = ax.contour(z500.longitude,z500.latitude,z500,levels=z500_levels,extend='both',colors='k',alpha=0.6, transform =ccrs.PlateCarree(),zorder=2,linewidths=2)
    ax.contour(z500.longitude,z500.latitude,z500,levels=[5150],extend='both',colors='k',alpha=1, transform =ccrs.PlateCarree(),zorder=2,linewidths=2.5)
    distri = distri.sel(latitude=u500.latitude,longitude=u500.longitude)
    u500, v500 = np.where(distri<10, np.nan, u500), np.where(distri<10, np.nan, v500)
    xd,yd = 14,6
    Q = ax.quiver(lons[::xd],lats[::yd],u500[::yd,::xd], v500[::yd,::xd],color="#33FD3D",pivot="middle",scale=120, width=0.005,transform=ccrs.PlateCarree(),alpha=1,zorder=1)
    # 添加 quiverkey（axes 右上角）
    ax.quiverkey(Q, 0.85, 0.92, 10, "10 m/s",coordinates="axes", color="#33FD3D",fontproperties={"size": 14})
    ax.add_feature(cfeature.LAND, edgecolor='black',facecolor='lightgray',linewidth=1,zorder=1)
    return C1
def draw_thickness(ax,var,p_values):
    p_level  = 0.01
    thickness,u_avg,v_avg = var.thickness.values,var.u_avg,var.v_avg
    thickness_p,wind_avg_p = p_values.thickness.values,p_values.wind_avg.values
    lons,lats = var.longitude,var.latitude
    #地理数据
    ax.add_feature(cfeature.LAND, edgecolor='black',facecolor='lightgray',linewidth=1,zorder=2)
    thickness[thickness_p>p_level]= np.nan
    basic_draw_sets(ax)
    C1 = ax.contourf(lons,lats,thickness,levels=10,extend='both',cmap= cmaps.precip4_diff_19lev_r,zorder=0, transform =ccrs.PlateCarree())#画等值线
    sig_mask =  wind_avg_p < p_level
    u_avg, v_avg = np.where(sig_mask, u_avg, np.nan), np.where(sig_mask, v_avg, np.nan)
    ax.barbs(lons,lats,u_avg, v_avg,regrid_shape=15,barb_increments={'half':1,'full':2,'flag':10},color='white',pivot='middle',transform=ccrs.PlateCarree(), alpha=1,sizes={'height':0.4},zorder=1)
    return C1
def draw_wind(ax,var,p_values):
    p_level  = 0.01
    u500,v500 = var.u500,var.u500
    wind_p = p_values.wind_500.values
    lons,lats = var.longitude,var.latitude
    wind = np.sqrt(u500**2+v500**2)
    #地理数据
    #u500_p.values = np.where(u500<0,0.99,u500_p)
    ax.add_feature(cfeature.LAND, edgecolor='black',facecolor='lightgray',linewidth=1,zorder=2)
    wind[wind_p>p_level]= np.nan
    basic_draw_sets(ax)
    C1 = ax.contourf(lons,lats,wind,levels=np.arange(-10,11,1),extend='both',cmap= cmaps.precip4_diff_19lev,zorder=0, transform =ccrs.PlateCarree())#画等值线
    return C1
colors = [  "#F7F3F3","#E60707"]
# 创建自定义colormap
gray_cmap_r = mcolors.LinearSegmentedColormap.from_list('custom_gray', colors, N=256)
def draw_sensible_heat(ax,var,p_values):
    basic_draw_sets(ax)
    p_level  = 0.01
    sensible_heat,latent_heat,thickness_gra = var.sensible_heat.values,gaussian_filter(var.latent_heat.values,3),gaussian_filter(var.thickness_gra.values,2)
    sensible_heat_p,latent_heat_p,thickness_gra_p = p_values.sensible_heat.values,p_values.latent_heat.values,p_values.thickness_gra.values
    lons,lats = var.longitude,var.latitude
    #地理数据
    sensible_heat[sensible_heat_p>p_level] = np.nan
    thickness_gra[thickness_gra_p>p_level] = np.nan
    thickness_gra[thickness_gra<1]  = np.nan
    #vort_adv[vort_adv<0]= np.nan
    C1 = ax.contourf(lons,lats,sensible_heat,levels=np.arange(0,55,5),extend='both',cmap= cmaps.cmocean_dense,zorder=0, transform =ccrs.PlateCarree())#画等值线
    CS = ax.contour(lons,lats,thickness_gra,levels=np.arange(2,7,1),extend='both',cmap=cmaps.MPL_YlOrRd,transform =ccrs.PlateCarree(),zorder=1,linewidths=2)
    ax.add_feature(cfeature.LAND, edgecolor='black',facecolor='lightgray',linewidth=1,zorder=1)
    ax.coastlines(color='black',linewidth=1)
    
    return C1

In [4]:
def wind_rose(ax,distances):
    all_dictretions = ['N','NNE', 'NE', 'ENE', 'E', 'ESE', 'SE', 'SSE', 'S', 'SSW', 'SW', 'WSW', 'W', 'WNW', 'NW', 'NNW']
    mean_distances = distances.groupby('direction')['mean_distance'].mean().reindex(all_dictretions, fill_value=0)
    direction_counts = distances['direction'].value_counts().reindex(all_dictretions, fill_value=0)/len(distances)*100
    color_list =["#a0cee4", "#5f9c80","#410E46"]
    colors = np.array([color_list[0]]*len(all_dictretions))
    colors[(mean_distances.values>=15)&(mean_distances.values<25)] = color_list[1]
    colors[mean_distances.values>=25] = color_list[2]
    
    # 绘图 
    sectors = 16
    sector_width = 360 / sectors
    # 创建极坐标图
    # 计算每个扇区的频率
    sector_angles = np.linspace(0, 2*np.pi, sectors, endpoint=False)
    
    # 绘制风玫瑰图
    bars = ax.bar(sector_angles, direction_counts.values, 
                  width=2*np.pi/sectors, alpha=1, 
                  color=colors, edgecolor='black',zorder=1)
    
    # 设置极坐标图的属性
    ax.set_theta_zero_location('N')  # 0°在顶部（北）
    ax.set_theta_direction(-1)       # 顺时针方向
    ax.set_rlabel_position(0)        # 距离标签位置
    
    # 设置方向标签 - 只显示N, S, E, W四个主要方向
    major_angles = [0]  # N, E, S, W
    ax.set_xticks(major_angles)
    # 设置径向刻度和标签
    ax.set_rlim(0, 20)
    ax.set_rticks(np.arange(0, 21, 5))
    ax.set_yticklabels([])
    
    ax.grid(linestyle='--',zorder=0,alpha=0.5)

In [ ]:
ori_ERA5_tracks = pd.read_excel(r"D:\Desktop\PLs_feature\new_track_infos\ERA5_tracks_smoothed.xlsx")
ori_ERA5_tracks['year'] = ori_ERA5_tracks.time.dt.year
ori_ERA5_tracks['month'] = ori_ERA5_tracks.time.dt.month
datasets = pd.read_excel(r'D:\Desktop\PLs_feature\feature_track\features_track.xlsx').set_index('ID')
PMCs_climate = np.loadtxt(r"D:\Desktop\PLs_feature\climate\PMCs_climate.txt",dtype=int)
life = ori_ERA5_tracks.value_counts('ID').sort_index()
#只保留PMCs
datasets = datasets.loc[(datasets.index.isin(PMCs_climate))]
datasets['ShearStrength'] = datasets['ShearStrength']*1000
whole_datasets = datasets.copy()

In [ ]:
class_folder = r'X:\classes'
folder = os.path.join(class_folder,'original')
all_class_names = [i.split('.')[0] for i in os.listdir(folder)]
class_dict = {}
for name in all_class_names:
    class_dict[name] = np.loadtxt(os.path.join(folder,name+'.txt')).astype('int')

In [ ]:
#PMCs频率
PMCs_tracks = ori_ERA5_tracks.loc[(ori_ERA5_tracks.ID.isin(PMCs_climate))].copy()
track_nums  = pd.DataFrame(PMCs_tracks.value_counts(['lat','lon'])).reset_index()
distri = xr.open_dataset(r"Z:\ERA5_data\surface2024\2000_1_surface.nc").msl[0]
lons,lats = distri.longitude,distri.latitude
#地理数据
lon_grid,lat_grid = np.meshgrid(lons,lats,indexing = 'xy')
lon_lat_points = pd.DataFrame(np.vstack([lon_grid.reshape(-1),lat_grid.reshape(-1)]).T,columns=['lon','lat'])
lat_lon_points = np.vstack([lat_grid.reshape(-1),lon_grid.reshape(-1)]).T
lat_lon_tree = BallTree(np.radians(lat_lon_points), leaf_size=20,metric = 'haversine')
SIC = xr.open_dataset(r"D:\Desktop\PLs_feature\ALL_siconc.nc").siconc
#---------------------------------
class_nums = np.array([len(class_dict[label]) for label in class_dict.keys()])
class_nums = class_nums/sum(class_nums)
class_distri_record,class_num_dict = {},{}
for label in class_dict.keys():
    PMCs_distri_record = []
    num_record = []
    print(label,len(class_dict[label]))
    for year in range(2000,2024):
        distri_year = distri.copy()
        distri_nums = np.zeros(len(lat_lon_points))
        winter = ((PMCs_tracks.year==year)&(PMCs_tracks.month.isin([11,12])))|((PMCs_tracks.year==year+1)&(PMCs_tracks.month.isin([1,2,3,4])))
        chosed = winter&(PMCs_tracks.ID.isin(class_dict[label]))
        if sum(chosed):
            in_200_points = lat_lon_tree.query_radius(np.radians(PMCs_tracks.loc[chosed,['lat','lon']].values), r=200/6371)
            for i in range(len(in_200_points)):
                distri_nums[in_200_points[i]] = distri_nums[in_200_points[i]]+1
        distri_year.values = distri_nums.reshape(distri.shape)
        distri_year['time'] = pd.Timestamp(year,1,1)
        #记录------------------------------------------
        PMCs_distri_record.append(distri_year)
        num_record.append(sum(chosed))
    class_distri_record[label] = xr.concat(PMCs_distri_record,dim='time').rename(label)
    class_num_dict[label] = np.array(num_record)


CAO-edge-dry 883
CAO-edge-wet 849
Cold-sea 568
Ice-edge 376
Strong-CAO-inner 1101
Warm-sea 715
Weak-CAO-dry 516
Weak-CAO-wet 556


In [ ]:
import cartopy.crs as ccrs
import matplotlib.path as mpath
import cartopy.feature as cfeature
def draw_distribution(ax,distri,title,pad):
    basic_draw_sets(ax)
    ax.set_title(title, fontsize=25, loc='center', fontweight='bold', pad=pad)
    ax.add_feature(cfeature.LAND, facecolor='gainsboro', edgecolor='dimgray', zorder=1)
    # 子模态使用相同的等值线设置
    C1 = ax.contourf(lons, lats, distri, 
                            levels=np.arange(100, 1700, 100), cmap=cmaps.WhiteYellowOrangeRed, 
                            transform=ccrs.PlateCarree(), zorder=0, extend='both')
    ax.contour(lons,lats,SIC,levels=[0.15],colors='b', transform =ccrs.PlateCarree(),linewidths=3,zorder=0)
    return C1
def creat_wind_rose_ax(fig, ax):
    pos = ax.get_position()
    size = pos.width * 0.3
    #(x1,y1)右上角
    wind_rose_x = pos.x1 - size - 0.03 * pos.width
    wind_rose_y = pos.y1- size - 0.4 * pos.width
    wax = fig.add_axes([wind_rose_x, wind_rose_y, size, size], projection='polar')
    return wax

In [ ]:
for class_name in class_dict.keys():
    whole_datasets.loc[class_dict[class_name],'with'] = class_name 
chosed_column = ['PBLH','CAODepth', 'ThickAdv', 'PVA','GrdThick', 
                 'SHF', 'LHF','CAPE','LHRR', 'LPV', 'SIC','SST', 'TropTheta', 'Thickness','Lifetime','Vort850','Vmax10m']#,'GrdPBLH','frac','V500','V925','CAO500','CAO700','ShearStrength'
# 计算均值并标准化
feature_describe = whole_datasets.groupby('with')[chosed_column].mean().loc[all_class_names]
feature_describe_normalized = (feature_describe - np.nanmean(feature_describe, axis=0)) / np.nanstd(feature_describe, axis=0)

In [ ]:
def plot_feature_comparison_abs(ax,category, df_raw, df_norm, top_n=12, sort=True):
    # 提取该行数据
    norm_series = df_norm.loc[category]        # 标准化值（可正可负）
    raw_series = df_raw.loc[category]          # 原始平均值
    # 合并并去除缺失值
    data = pd.DataFrame({'norm': norm_series, 'raw': raw_series}).dropna()
    # 添加绝对值列用于排序和高度
    data['abs_norm'] = data['norm'].abs()
    # 确定特征顺序
    if sort:
        data = data.sort_values('abs_norm', ascending=False)
    else:
        # 保持 df_raw 的列顺序
        col_order = [c for c in df_raw.columns if c in data.index]
        data = data.reindex(col_order)
    # 取前 top_n 个
    #确保CAODepth必须出现
    chosed_feature = list(data.index[:top_n])
    #if 'CAODepth' not in chosed_feature:chosed_feature[-1] = 'CAODepth'
    data = data.loc[chosed_feature]
    plt.setp(ax.get_xticklabels(), rotation=-20, fontsize=20)
    plt.setp(ax.get_yticklabels(), rotation=0, fontsize=20)
    # 绘制柱子：高度 = abs_norm，颜色由 norm 的正负决定
    abs_vals = data['abs_norm'].values
    colors = ["#B80202" if v > 0 else "#3102A0" for v in data['norm'].values]
    bars = ax.bar(data.index, abs_vals, color=colors,width=0.8, edgecolor='black', linewidth=1.2)
    # y轴标签（绝对值无单位，仍称标准化绝对值的标度）
    ax.set_ylabel('|Standardized Value|', fontsize=25)
    #ax.set_xlabel('Features', fontsize=25)
    #ax.set_ylim(0,2.5)
    ax.set_title(f'(a) Mean Feature', fontsize=25, fontweight='bold',loc='left')
    # 科研风格：隐藏上、右边框，添加网格线
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)
    # 在柱顶标原始标准化值（带符号，保留两位小数）
    y_min, y_max = ax.get_ylim()
    y_range = y_max - y_min
    offset = 0.005 * y_range   # 偏移量，避免与柱顶重叠
    '''
    for bar, feature in zip(bars, data['raw']):
        height = bar.get_height()
        # 始终在柱顶上方标注（因为高度为正）
        label_y = height + offset
        ax.text(bar.get_x() + bar.get_width()/2., label_y,
                f'{feature:.2f}', ha='center', va='bottom', fontsize=20,
                color='black')
    '''
    return ax

In [ ]:
'''
import os
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from tqdm import tqdm

# 创建输出目录
output_dir = r'D:\Desktop\PLs_feature\integrate_figures-split'
os.makedirs(output_dir, exist_ok=True)

proj = ccrs.LambertConformal(central_longitude=16)

for chosed_pmcs in tqdm(feature_describe.index):
    # -------------------- 1. 折线图（特征对比，无 colorbar）--------------------
    fig_line, ax_line = plt.subplots(figsize=(12, 6), dpi=300)
    ax_line = plot_feature_comparison_abs(ax_line, chosed_pmcs, feature_describe,
                                          feature_describe_normalized, top_n=14, sort=True)
    #ax_line.set_title(f'Feature comparison for {chosed_pmcs}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{chosed_pmcs}_global_feature_comparison.png'),
                dpi=300, bbox_inches='tight')
    plt.close(fig_line)

    # -------------------- 2. 分布图（带 colorbar）--------------------
    fig_dist, ax_dist = plt.subplots(figsize=(8, 6), dpi=300, subplot_kw={'projection': proj})
    C_dist = draw_distribution(ax_dist,
                               class_distri_record[chosed_pmcs].sum('time'),
                               f'distribution ({len(class_dict[chosed_pmcs]) / len(PMCs_climate) * 100:.1f}%)',
                               pad=10)
    cbar_dist = fig_dist.colorbar(C_dist, ax=ax_dist, orientation='horizontal',
                                  fraction=0.05, pad=0.05, label="times within 200km radius")
    cbar_dist.ax.tick_params(labelsize=12)
    ax_dist.set_title(f'{chosed_pmcs} - Distribution', fontsize=16, fontweight='bold')
    plt.savefig(os.path.join(output_dir, f'{chosed_pmcs}_global_distribution.png'),
                dpi=300, bbox_inches='tight')
    plt.close(fig_dist)

    # -------------------- 3. 两个区域的子图 --------------------
    for i, chosed_area in enumerate(['Nordic', 'Irminger']):
        # ----- 3.1 CAO 图（冷空气爆发）-----
        fig_cao, ax_cao = plt.subplots(figsize=(8, 6), dpi=300, subplot_kw={'projection': proj})
        distri, propation, ano_ds, p_ds = load_feature_data(['CAO_depth', 'theta_trop', 'u925', 'v925'],
                                                            f'{chosed_pmcs}({chosed_area})', 'whole')
        C_cao = draw_CAO(ax_cao, distri, ano_ds, p_ds)
        cbar_cao = fig_cao.colorbar(C_cao, ax=ax_cao, orientation='horizontal',
                                    fraction=0.05, pad=0.05, label='CAO depth (hPa)')
        cbar_cao.ax.tick_params(labelsize=12)
        ax_cao.set_title(f'CAO Depth & TropTheta & Wind925hPa ({chosed_area})',
                         fontsize=14, fontweight='bold')
        plt.savefig(os.path.join(output_dir, f'{chosed_pmcs}_{chosed_area}_CAO.png'),
                    dpi=300, bbox_inches='tight')
        plt.close(fig_cao)

        # ----- 3.2 水汽图（含水汽、500hPa风场、风玫瑰图）-----
        fig_q, ax_q = plt.subplots(figsize=(8, 6), dpi=300, subplot_kw={'projection': proj})
        _, propation, ano_ds, p_ds = load_feature_data(['q', 'siconc'],
                                                       f'{chosed_pmcs}({chosed_area})', 'whole')
        mean_vars = load_feature_mean(['u500', 'v500', 'z500'], chosed_pmcs+f'({chosed_area})', 'whole')
        C_q = draw_q(ax_q, mean_vars, ano_ds, distri, p_ds)   # 注意此处 distri 沿用 CAO 中的值，原代码如此
        # 添加风玫瑰图
        rose_ax = creat_wind_rose_ax(fig_q, ax_q)
        wind_rose(rose_ax, propation)
        cbar_q = fig_q.colorbar(C_q, ax=ax_q, orientation='horizontal',
                                fraction=0.05, pad=0.05, label='q (g kg s$^{-1}$)')
        cbar_q.ax.tick_params(labelsize=12)
        ax_q.set_title(f'Q & Z500 & Wind500hPa ({chosed_area})',
                       fontsize=14, fontweight='bold')
        plt.savefig(os.path.join(output_dir, f'{chosed_pmcs}_{chosed_area}_Q.png'),
                    dpi=300, bbox_inches='tight')
        plt.close(fig_q)

        # ----- 3.3 感热图（感热/潜热 + 厚度梯度）-----
        fig_shf, ax_shf = plt.subplots(figsize=(8, 6), dpi=300, subplot_kw={'projection': proj})
        _, _, ano_ds, p_ds = load_feature_data(['sensible_heat', 'latent_heat', 'thickness_gra'],
                                               f'{chosed_pmcs}({chosed_area})', 'whole')
        C_shf = draw_sensible_heat(ax_shf, ano_ds, p_ds)
        cbar_shf = fig_shf.colorbar(C_shf, ax=ax_shf, orientation='horizontal',
                                    fraction=0.05, pad=0.05, label='sensible heat (W m s$^{-2}$)')
        cbar_shf.ax.tick_params(labelsize=12)
        ax_shf.set_title(f'SHF & GrdThick ({chosed_area})',
                         fontsize=14, fontweight='bold')
        plt.savefig(os.path.join(output_dir, f'{chosed_pmcs}_{chosed_area}_SHF.png'),
                    dpi=300, bbox_inches='tight')
        plt.close(fig_shf)
'''

'\nimport os\nimport matplotlib.pyplot as plt\nimport cartopy.crs as ccrs\nfrom tqdm import tqdm\n\n# 创建输出目录\noutput_dir = r\'D:\\Desktop\\PLs_feature\\integrate_figures-split\'\nos.makedirs(output_dir, exist_ok=True)\n\nproj = ccrs.LambertConformal(central_longitude=16)\n\nfor chosed_pmcs in tqdm(feature_describe.index):\n    # -------------------- 1. 折线图（特征对比，无 colorbar）--------------------\n    fig_line, ax_line = plt.subplots(figsize=(12, 6), dpi=300)\n    ax_line = plot_feature_comparison_abs(ax_line, chosed_pmcs, feature_describe,\n                                          feature_describe_normalized, top_n=14, sort=True)\n    #ax_line.set_title(f\'Feature comparison for {chosed_pmcs}\', fontsize=16, fontweight=\'bold\')\n    plt.tight_layout()\n    plt.savefig(os.path.join(output_dir, f\'{chosed_pmcs}_global_feature_comparison.png\'),\n                dpi=300, bbox_inches=\'tight\')\n    plt.close(fig_line)\n\n    # -------------------- 2. 分布图（带 colorbar）--------------------\n  

In [ ]:
for chosed_pmcs in tqdm(feature_describe.index):
    proj = ccrs.LambertConformal(central_longitude=16)
    fig = plt.figure(figsize=(26, 20), dpi=300)
    gs = GridSpec(5,3, height_ratios=[1,0.03,1,1,0.03], wspace=0.15, hspace=0.16)
    letters = ['(c)', '(d)', '(e)', '(f)', '(g)', '(h)', '(i)', '(j)', '(k)', '(l)','(m)', '(n)', '(o)', '(p)', '(q)', '(r)', '(s)', '(t)', '(u)', '(v)', '(w)']
    #折线图
    ax0 = fig.add_subplot(gs[0, :2])
    ax0 = plot_feature_comparison_abs(ax0,chosed_pmcs, feature_describe, feature_describe_normalized, top_n=14, sort=True)
    #分布
    ax1 = fig.add_subplot(gs[0, 2], projection=proj)
    C1 = draw_distribution(ax1,class_distri_record[chosed_pmcs].sum('time'),f'(b) distribution ({len(class_dict[chosed_pmcs])/len(PMCs_climate)*100:.1f}%)',pad=10)
    cax1 =  fig.add_subplot(gs[1, 2]) 
    cbar1 = fig.colorbar(C1, cax=cax1,orientation='horizontal')
    cbar1.set_label("times within 200km radius", fontsize=20, labelpad=2)
    cax1.tick_params(axis='x', labelsize=20)
    #冷空气爆发
    for i,chosed_area in enumerate(['Nordic','Irminger']):
        ax2 = fig.add_subplot(gs[2+i, 0], projection=proj) 
        distri, propation, ano_ds, p_ds = load_feature_data(['CAO_depth','theta_trop','u925','v925'], chosed_pmcs+f'({chosed_area})', 'whole')
        C1 = draw_CAO(ax2,distri, ano_ds, p_ds)
        if i == 0:
            title = f"CAODepth&TropTheta&Wind925hPa"+'\n'+letters[3*i]+ f' {chosed_area}'
        else:
            title = letters[3*i]+ f' {chosed_area}'
        ax2.set_title(title, fontsize=25, fontweight='bold', pad=10)

        #水汽
        ax3 = fig.add_subplot(gs[2+i, 1], projection=proj) 
        _, propation, ano_ds, p_ds = load_feature_data(['q','siconc'], chosed_pmcs+f'({chosed_area})', 'whole')
        mean_vars = load_feature_mean(['u500','v500','z500'], chosed_pmcs+f'({chosed_area})', 'whole')
        C2 = draw_q(ax3,mean_vars,ano_ds,distri, p_ds)
        rose_ax = creat_wind_rose_ax(fig, ax3)
        wind_rose(rose_ax,propation)
        if i == 0:
            title = f"Q&Z500&Wind500hPa"+'\n'+letters[3*i+1]+ f' {chosed_area}'
        else:
            title = letters[3*i+1]+ f' {chosed_area}'
        ax3.set_title(title, fontsize=25, fontweight='bold', pad=10)
        #感热
        ax4 = fig.add_subplot(gs[2+i, 2], projection=proj) 
        _, _, ano_ds, p_ds = load_feature_data(['sensible_heat','latent_heat','thickness_gra'], chosed_pmcs+f'({chosed_area})', 'whole')
        C3 = draw_sensible_heat(ax4,ano_ds,p_ds)
        if i == 0:
            title = f"SHF&GrdThick"+'\n'+letters[3*i+2]+ f' {chosed_area}'
        else:
            title = letters[3*i+2]+ f' {chosed_area}'
        ax4.set_title(title, fontsize=25, fontweight='bold', pad=10)
    #CAX
    cax2 = fig.add_subplot(gs[4, 0]) 
    cb = fig.colorbar(C1, cax=cax2,orientation='horizontal')
    cb.set_label('CAO depth (hPa)', size=20, labelpad=2)
    cax2.tick_params(axis='x', labelsize=20)
    cax3 = fig.add_subplot(gs[4, 1]) 
    cb = fig.colorbar(C2, cax=cax3,orientation='horizontal')
    cb.set_label('q (g kg s$^{-1}$)', size=20, labelpad=2)
    cax3.tick_params(axis='x', labelsize=20)
    cax4 = fig.add_subplot(gs[4, 2])
    cb = fig.colorbar(C3, cax=cax4,orientation='horizontal')
    cb.set_label('sensible heat (W m s$^{-2}$)', size=20, labelpad=2)
    cax4.tick_params(axis='x', labelsize=20)
    for cax in [cax1,cax2,cax3,cax4]:
        pos = cax.get_position()        # 返回 (x0, y0, width, height)
        # 向上移动 0.02 (可根据需要调整)
        new_y0 = pos.y0 + 0.015
        cax.set_position([pos.x0, new_y0, pos.width, pos.height])
    #第一排往上提
    for ax in [ax0,ax1,cax1]:
        pos = ax.get_position()        # 返回 (x0, y0, width, height)
        # 向上移动 0.02 (可根据需要调整)
        new_y0 = pos.y0 + 0.025
        ax.set_position([pos.x0, new_y0, pos.width, pos.height])
    fig.suptitle(chosed_pmcs, fontsize=40, fontweight='bold', y=0.96)
    plt.savefig(f'D:\Desktop\PLs_feature\integrate_figures\{chosed_pmcs}.png',dpi=300,bbox_inches='tight')
    plt.close()

100%|██████████| 8/8 [03:09<00:00, 23.66s/it]
